In [ ]:
import itertools
from ast import literal_eval
import os

import numpy as np
import pandas as pd
from scipy.integrate import trapezoid

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from mg_sampling import PROJECT_DIR
from mg_sampling.constants import RT
from mg_sampling.tools import read_plumed, read_plumed_fes, range_to_edges, edges_to_range, finitemin, finitemax, block_analysis, read_wsv, demux
from mg_sampling.bias import bias_fn

LABELS = {
    'amber_tip3p': "Amber/TIP3P",
    'mMg_tip3p': "microMg/TIP3P",
    'nMg_tip3p': "nanoMg/TIP3P", 
    'mMg_opc': "microMg/OPC",
    'ecc_spce': "AmberECC/SPCE",
}

FORCEFIELDS_ARCHIVE = ['amber_tip3p', 'mMg_tip3p', 'nMg_tip3p', 'mMg_opc', 'ecc_spce']
FORCEFIELDS = ['mMg_tip3p']

### Evolution of the number of inner-bound Mg²⁺ (Replex vs. PlainMD)

In [ ]:
n_inner_pdb = 14

system = '7ez0'
ff = 'mMg_tip3p'
replicate = 0
nrep = 8

fig, ax = plt.subplots(figsize=(5, 4))

# Replex
d_mg_op_dmx = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

d_mg_op = demux(d_mg_op_dmx, repindex_file=PROJECT_DIR / f'replex/{system}/{ff}/replicate{replicate}/rep00/bias_index.xvg')

time = d_mg_op[0, :, 0] / 1000  # ns
is_bound = (d_mg_op[0, :, 2:] < 0.28)

ax.plot(time, is_bound.sum(axis=-1), color='tomato', lw=1, label='μMg (REx)')

# Plain MD

ff = 'mMg_tip3p_hard'
nrep = 4

d_mg_op = [
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
]
n_frames = max([len(x) for x in d_mg_op])
d_mg_op = np.stack([x[:n_frames] for x in d_mg_op])

time = d_mg_op[0, :, 0] / 1000  # ns
is_bound = (d_mg_op[:, :, 2:] < 0.28)


n_bound_avg = is_bound.sum(axis=-1).mean(axis=0)
n_bound_err = 1.96 * is_bound.sum(axis=-1).std(axis=0) / np.sqrt(nrep - 1)
ax.plot(time, n_bound_avg, color='mediumpurple', lw=1, label='μMg (MD)')
ax.fill_between(time, n_bound_avg - n_bound_err, n_bound_avg + n_bound_err, fc='mediumpurple', alpha=0.25)


ff = 'nMg_tip3p_hard'
nrep = 4

d_mg_op = np.stack([
    np.insert(
        np.loadtxt(PROJECT_DIR / f'plain_md/{system}/{ff}/rep{i:02d}/dist_MG_O_Phosphate.xvg', comments=['@', '#']),
        1,
        i,
        axis=-1,
    )
    for i in range(nrep)
])

time = d_mg_op[0, :, 0] / 1000  # ns
is_bound = (d_mg_op[:, :, 2:] < 0.28)


n_bound_avg = is_bound.sum(axis=-1).mean(axis=0)
n_bound_err = 1.96 * is_bound.sum(axis=-1).std(axis=0) / np.sqrt(nrep - 1)
ax.plot(time, n_bound_avg, color='gray', lw=1, label='nMg (MD)')
ax.fill_between(time, n_bound_avg - n_bound_err, n_bound_avg + n_bound_err, fc='gray', alpha=0.25)


ax.axhline(n_inner_pdb, ls='dashed', lw=1, color='silver')
ax.text(500, n_inner_pdb - 1, 'PDB', style='italic', color='silver', ha='left', va='top')

ax.legend(frameon=False)
ax.set_xlim(-10, 600)
ax.set_ylim(0, 100)
ax.set_xlabel('Time [ns]')
ax.set_ylabel(r'Mg$^{\text{2+}}$ inner-bound to $\mathrm{O_P}$')

plt.savefig('figures/inner_mg_op_timeseries.pdf')